In [1]:
import os
import h5py
import numpy as np
import pandas as pd
import re

In [5]:
def list_h5_files(directory):
    """
    List all H5 files in the given directory, sorted numerically.
    """
    files = [f for f in os.listdir(directory) if f.endswith('.h5')]
    
    # Sort by the number after 'hyb_' in the filename
    files.sort(key=lambda x: int(re.search(r"hyb_(\d+)_", x).group(1)))
    
    return [os.path.join(directory, f) for f in files]

def merge_h5_files(directory, output_file):
    """
    Merge all H5 files in the given directory into a single H5 file.
    """
    file_paths = list_h5_files(directory)
    N_points, C, W, H, D = 200, 3, 2048, 2048, 77 #REMEMBER TO CHANGE "D" TO THE ACTUAL Z NUMBER -1
    T = len(file_paths)

    with h5py.File(output_file, "a") as out_h5:
        for i, file_path in enumerate(file_paths):
            with h5py.File(file_path, "r") as in_h5:
                print(file_path)
                image_16bit_all_channels = np.array(in_h5['data'])  # Assuming 'data' is your original 16-bit image data
                image_8bit_all_channels = []
                for image_16bit in image_16bit_all_channels:
                    # Scale the 16-bit image data to 8-bit
                    quantiles = np.quantile(image_16bit, [0,1]) #adjust as desired
                    percentile_01, percentile_99 = quantiles[0], quantiles[1]
                    image_16bit[image_16bit<percentile_01] = 0
                    image_16bit[image_16bit>percentile_99] = 65535
                    # image_8bit = (((image_16bit - percentile_01) / (percentile_99 - percentile_01)) * 255).astype(np.uint8)
                    image_8bit = ((image_16bit - image_16bit.min()) / (image_16bit.max() - image_16bit.min()) * 255).astype(np.uint8)
                    # Now, applying the additional thresholds directly based on your requirements
                    image_8bit[image_16bit == 0] = 0  # Values below 0.2 quantile become 0
                    image_8bit[image_16bit == 65535] = 255  # Values above 0.99 quantile become 255
                    image_8bit_all_channels.append(image_8bit)
                image_8bit_all_channels = np.array(image_8bit_all_channels)
                # Create a dataset for the 8-bit image
                ds = out_h5.create_dataset(f"{i}/frame", shape=image_8bit_all_channels.shape, dtype=np.uint8, compression="lzf")
                ds[...] = image_8bit_all_channels

        # Set global attributes based on your specifications
        out_h5.attrs["N_points"] = N_points
        points = np.zeros((T, N_points + 1, 3)).astype(np.float32)
        points[:] = np.nan
        points *= np.array([W, H, D])[None, None, :]

        ds = out_h5.create_dataset("points", points.shape, points.dtype)
        ds[...] = points

        out_h5.attrs["T"] = T
        out_h5.attrs["C"] = C
        out_h5.attrs["W"] = W
        out_h5.attrs["H"] = H
        out_h5.attrs["D"] = D
        out_h5.attrs["description"] = "example data"

    print("Compiled H5 made")


In [8]:
if __name__ == "__main__":
    directory =r"C:\Users\josed\OneDrive\Documents\DATA\Zadjusted\adjusted_tiff\raw H5 files\pos1" #Update this path to the directory containing your H5 files
    output_file =r"D:\data\malemarkers\processed\MG2_Pool1\CompiledH5\MG2pool1\pos1_remake.h5" #Add name of the file at the end and use .h5
    merge_h5_files(directory, output_file)

C:\Users\josed\OneDrive\Documents\DATA\Zadjusted\adjusted_tiff\raw H5 files\pos1\hyb_000_pos1.h5
C:\Users\josed\OneDrive\Documents\DATA\Zadjusted\adjusted_tiff\raw H5 files\pos1\hyb_001_pos1.h5
C:\Users\josed\OneDrive\Documents\DATA\Zadjusted\adjusted_tiff\raw H5 files\pos1\hyb_002_pos1.h5
C:\Users\josed\OneDrive\Documents\DATA\Zadjusted\adjusted_tiff\raw H5 files\pos1\hyb_003_pos1.h5
C:\Users\josed\OneDrive\Documents\DATA\Zadjusted\adjusted_tiff\raw H5 files\pos1\hyb_004_pos1.h5
C:\Users\josed\OneDrive\Documents\DATA\Zadjusted\adjusted_tiff\raw H5 files\pos1\hyb_005_pos1.h5
C:\Users\josed\OneDrive\Documents\DATA\Zadjusted\adjusted_tiff\raw H5 files\pos1\hyb_006_pos1.h5
C:\Users\josed\OneDrive\Documents\DATA\Zadjusted\adjusted_tiff\raw H5 files\pos1\hyb_007_pos1.h5
C:\Users\josed\OneDrive\Documents\DATA\Zadjusted\adjusted_tiff\raw H5 files\pos1\hyb_008_pos1.h5
C:\Users\josed\OneDrive\Documents\DATA\Zadjusted\adjusted_tiff\raw H5 files\pos1\hyb_009_pos1.h5
C:\Users\josed\OneDrive\Docume

In [49]:
import os
import h5py
import numpy as np
import pandas as pd

In [51]:
# Input file paths
file_path_1 = r"C:\Users\josed\Desktop\DATA HERE PLEASE\CompiledPos3.h5"
file_path_2 = r"C:\Users\josed\Desktop\DATA HERE PLEASE\CompiledPos4_test2.h5"
file_path_3 = r"C:\Users\josed\Desktop\DATA HERE PLEASE\CompiledPos11.h5"

# Output file path
merged_file_path = r"C:\Users\josed\Desktop\DATA HERE PLEASE\MergedData.h5"

# Create a new HDF5 file for merging
with h5py.File(merged_file_path, "w") as out_file:
    with h5py.File(file_path_1, "r") as in_1:
        out_file.create_dataset("0/frame", data=in_1["5/frame"][:])

    with h5py.File(file_path_2, "r") as in_2:
        out_file.create_dataset("1/frame", data=in_2["5/frame"][:])

    with h5py.File(file_path_3, "r") as in_3:
        out_file.create_dataset("2/frame", data=in_3["5/frame"][:])

print("HDF5 files merged successfully!")

HDF5 files merged successfully!


In [90]:
# Define cropping dimensions (height, width)

x = 800
y = 800

# Input file paths
file_path_1 = r"C:\Users\josed\Desktop\DATA HERE PLEASE\CompiledPos3.h5"
file_path_2 = r"C:\Users\josed\Desktop\DATA HERE PLEASE\CompiledPos4_test2.h5"
file_path_3 = r"C:\Users\josed\Desktop\DATA HERE PLEASE\CompiledPos11.h5"

# Output file path
merged_file_path = r"C:\Users\josed\Desktop\DATA HERE PLEASE\CroppedData.h5"

# Function to crop an image
def crop_image(n, image, x_start, x_end, y_start, y_end):
    if n == 1:
        return image[:,347:347+x,287:287+y,25:50]
    elif n == 2:
        return image[:,191:191+x,342:342+y:,25:50]
    elif n == 3:
        return image[:,412:412+x,1053:1053+y,25:50]

# Create a new HDF5 file for merging
with h5py.File(merged_file_path, "w") as out_file:
    with h5py.File(file_path_1, "r") as in_1:
        cropped_1 = crop_image(1, in_1["5/frame"][:], crop_x_start, crop_x_end, crop_y_start, crop_y_end)
        out_file.create_dataset("0/frame", data=cropped_1)

    with h5py.File(file_path_2, "r") as in_2:
        cropped_2 = crop_image(2, in_2["5/frame"][:], crop_x_start, crop_x_end, crop_y_start, crop_y_end)
        out_file.create_dataset("1/frame", data=cropped_2)

    with h5py.File(file_path_3, "r") as in_3:
        cropped_3 = crop_image(3, in_3["5/frame"][:], crop_x_start, crop_x_end, crop_y_start, crop_y_end)
        out_file.create_dataset("2/frame", data=cropped_3)

print("HDF5 files merged successfully with cropped images!")

HDF5 files merged successfully with cropped images!


In [91]:
h5 = h5py.File(merged_file_path, "a")

# Adding attributes
h5.attrs["N_points"] = 200
h5.attrs["T"] = 3
h5.attrs["C"] = 3
h5.attrs["W"] = 800
h5.attrs["H"] = 800
h5.attrs["D"] = 24
h5.attrs["description"] = "example data"

h5.close()


In [70]:
with h5py.File(merged_file_path, "r") as f:
    print("File Attributes:")
    for attr in f.attrs:
        print(f"{attr}: {f.attrs[attr]}")

File Attributes:
C: 3
D: 15
H: 100
N_points: 200
T: 3
W: 100
description: example data
